# 00 · Synthetic data — checking notebook (dev only, temporary)

**What this is for.** Look at the synthetic feature mart after each build step and decide whether it
looks like a plausible book. The spec and the build plan are in `docs/components/test_harness.md`.
This notebook is temporary: the guided EDA notebook replaces it once EDA exists.

**How to use it.**
1. Set the preset and size in the next cell. `canonical` is 100k clients; on an 8 GB laptop use
   `N_CLIENTS = 50_000` if memory gets tight. `unit` (5k) is what the tests use.
2. Run all cells. First look at the check table: every row must pass. A failing row is a generator
   bug, not something to wave through.
3. Then use the summaries and charts for the step's gate. Step 1 (sections 1–7): *does the
   population look like our book?* Step 2 (section 8): *do holdings, rates and dynamics look right?* Segment mix; nationality, employment and emirate by subsegment; ages, income
   levels and missing income; relationship and residence lengths and the gap between them; turnover
   per profile.

The oracle frame holds hidden truth (latent traits now, true probabilities later). It is shown at the
end for checking only; it is never a feature.

In [ ]:
import pandas as pd
import plotly.express as px

from synthetic.checks_holdings import check_holdings
from synthetic.checks_population import check_population
from synthetic.generate import generate_mart

PRESET = "canonical"  # "unit" for a quick look
N_CLIENTS = None  # None = the preset's size; 50_000 halves memory on an 8 GB laptop
SEED = 0

mart, oracle, truth = generate_mart(PRESET, n_clients=N_CLIENTS, seed=SEED)
first = mart[mart["snapshot_month"] == mart["snapshot_month"].min()]
ORDER = list(truth["spec"]["subsegments"])  # wealthiest first
PROFILES = ["private", "priority", "affluent", "mass"]
profile = mart["subsegment"].map(lambda s: truth["spec"]["subsegments"][s]["profile"])
n_clients, n_months = mart["client_id"].nunique(), mart["snapshot_month"].nunique()
print(f"{len(mart):,} rows · {n_clients:,} clients · {n_months} months")

## 1. Check table
Every row must pass. `expected` vs `realised` with the tolerance; the full table is below the failures.

In [ ]:
checks = check_population(mart, oracle, truth)
print(f"{len(checks)} checks, {(~checks['passed']).sum()} failed")
checks[~checks["passed"]]

In [ ]:
checks.groupby("check", sort=False)["passed"].agg(["size", "sum"]).rename(
    columns={"size": "rows", "sum": "passed"}
)

In [ ]:
checks

## 2. Columns at a glance (first month)
Null share, number of distinct values and quantiles per column.

In [ ]:
summary = pd.DataFrame(
    {
        "dtype": first.dtypes.astype(str),
        "null share": first.isna().mean(),
        "distinct": first.nunique(),
    }
)
summary.join(first.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.drop(columns="count"))

## 3. Segments and subsegments (first month)
Check: shares, ages, income levels and how often income is missing, salary transfers.

In [ ]:
first.groupby("subsegment").agg(
    segment=("segment", "first"),
    clients=("client_id", "size"),
    share=("client_id", lambda s: len(s) / len(first)),
    mean_age=("age", "mean"),
    median_income=("income", "median"),
    income_missing=("income", lambda s: s.isna().mean()),
    salary_transfer=("salary_transfer_flag", "mean"),
    median_relationship_years=("relationship_days", lambda s: s.median() / 365.25),
    median_residence_years=("residence_days", lambda s: s.median() / 365.25),
).reindex(ORDER).round(3)

## 4. Nationality, employment and emirate by subsegment (row shares, first month)

In [ ]:
pd.crosstab(first["subsegment"], first["nationality_group"], normalize="index").reindex(
    ORDER
).round(3)

In [ ]:
pd.crosstab(first["subsegment"], first["employment_status"], normalize="index").reindex(
    ORDER
).round(3)

In [ ]:
pd.crosstab(first["subsegment"], first["emirate"], normalize="index").reindex(ORDER).round(3)

In [ ]:
first["nationality"].value_counts(normalize=True).head(20).round(4)

## 5. Distributions

In [ ]:
px.histogram(
    first,
    x="age",
    color="segment",
    barmode="overlay",
    histnorm="probability density",
    nbins=60,
    title="Age by segment",
)

In [ ]:
px.box(
    first.dropna(subset=["income"]),
    x="subsegment",
    y="income",
    log_y=True,
    points=False,
    category_orders={"subsegment": ORDER},
    title="Declared monthly income by subsegment (log scale; missing incomes left out)",
)

In [ ]:
px.histogram(
    first.assign(relationship_years=first["relationship_days"] / 365.25),
    x="relationship_years",
    color="segment",
    barmode="overlay",
    histnorm="probability density",
    nbins=60,
    title="Relationship length (years) by segment",
)

In [ ]:
residents = first.dropna(subset=["residence_days"])
gap_years = (residents["residence_days"] - residents["relationship_days"]) / 365.25
px.histogram(
    gap_years,
    nbins=80,
    title="Years between arrival in the country and relationship start (residents)",
)

## 6. Over time
Clients on the books, new-to-bank entries and exits per month; the subsegment mix should stay flat.

In [ ]:
turnover = pd.DataFrame(truth["turnover"])
rates = turnover.assign(
    entry_rate=turnover["entries"] / turnover["clients_before"],
    exit_rate=turnover["exits"] / turnover["clients_before"],
).melt(id_vars=["month", "profile"], value_vars=["entry_rate", "exit_rate"], var_name="rate")
px.line(
    rates,
    x="month",
    y="value",
    color="profile",
    line_dash="rate",
    title="Monthly entry and exit rate per profile",
)

In [ ]:
mix = pd.crosstab(mart["snapshot_month"], mart["subsegment"], normalize="index")[ORDER]
px.line(mix, title="Subsegment share per month")

In [ ]:
some_clients = mart["client_id"].drop_duplicates().sample(5, random_state=0)
mart[mart["client_id"].isin(some_clients)].sort_values(["client_id", "snapshot_month"]).groupby(
    "client_id"
).head(3)

## 7. Hidden truth (oracle) — for checking only
Mean latent traits per subsegment in the last month; wealth should fall from Upper Private to LM.

In [ ]:
last_month = mart["snapshot_month"].max()
latest = mart.loc[mart["snapshot_month"] == last_month, ["client_id", "subsegment"]].merge(
    oracle[oracle["snapshot_month"] == last_month], on="client_id"
)
latest.groupby("subsegment")[[c for c in oracle.columns if c.startswith("z_")]].mean().reindex(
    ORDER
).round(2)

## 8. Step 2: holdings and states
Check table first, then shares by profile (first vs last month: they should stay flat), statuses,
loans, events a month, the income-threshold jump, the age shape of personal loans and the oracle's
chances. Spec: the product table in `docs/components/test_harness.md`; values in
`synthetic/src/synthetic/products.py`.

In [ ]:
holding_checks = check_holdings(mart, oracle, truth)
print(f"{len(holding_checks)} checks, {(~holding_checks['passed']).sum()} failed")
holding_checks[~holding_checks["passed"]]

In [ ]:
holding_checks.groupby("check", sort=False)["passed"].agg(["size", "sum"]).rename(
    columns={"size": "rows", "sum": "passed"}
)

In [ ]:
flags = [
    "casa_flag",
    "cc_flag",
    "td_flag",
    "wealth_digital_flag",
    "wealth_trad_flag",
    "ebanking_flag",
    "fx_active_flag",
    "digital_active_flag",
]
loans = ["loan_personal_cnt", "loan_auto_cnt", "loan_home_cnt"]
months = sorted(mart["snapshot_month"].unique())
held = mart[flags].join(mart[loans].gt(0)).assign(profile=profile, month=mart["snapshot_month"])
shares = (
    held[held["month"].isin([months[0], months[-1]])].groupby(["month", "profile"]).mean().round(3)
)
shares.unstack("month").reindex(PROFILES)

In [ ]:
status_cols = ["td_status", "wealth_digital_status", "wealth_trad_status"]
first_month = mart["snapshot_month"] == months[0]
pd.concat(
    {
        c: pd.crosstab(profile[first_month], mart.loc[first_month, c], normalize="index")
        for c in status_cols
    },
    axis=1,
).reindex(PROFILES).round(3)

In [ ]:
pd.concat({c: mart.loc[first_month, c].value_counts().sort_index() for c in loans}, axis=1).fillna(
    0
).astype(int)

In [ ]:
px.histogram(
    mart.loc[first_month, "loan_personal_maturity_months"].dropna().astype(int),
    nbins=48,
    title="Personal loans: months to the nearest maturity (first month)",
)

In [ ]:
by_client = mart.sort_values(["client_id", "snapshot_month"])
same = by_client["client_id"].eq(by_client["client_id"].shift())
per_100k = 100_000 / mart.groupby("snapshot_month").size()
events = {}
for c in ["cc_flag", "td_flag", "wealth_digital_flag", "wealth_trad_flag", "ebanking_flag"]:
    gained = by_client[c] & ~by_client[c].shift(fill_value=False) & same
    events[c.removesuffix("_flag")] = gained.groupby(by_client["snapshot_month"]).sum()
for c in loans:
    events[c.removesuffix("_cnt")] = (
        ((by_client[c].diff() > 0) & same).groupby(by_client["snapshot_month"]).sum()
    )
events = pd.DataFrame(events).mul(per_100k, axis=0).iloc[1:]
px.line(events, title="Events a month per 100k clients (adoptions, new loans; incl. onboarding)")

In [ ]:
non_holders = by_client[
    same & ~by_client["cc_flag"].shift(fill_value=False) & (by_client["relationship_days"] >= 92)
]
bands = pd.cut(
    non_holders["income"], [0, 2_000, 3_000, 4_000, 4_500, 5_000, 5_500, 6_000, 8_000, 12_000, 1e9]
)
px.bar(
    non_holders.groupby(bands, observed=True)["cc_flag"]
    .mean()
    .rename("adoption rate")
    .reset_index()
    .astype({"income": str}),
    x="income",
    y="adoption rate",
    title="Credit card: monthly adoption rate by declared income (the 5k threshold)",
)

In [ ]:
px.line(
    mart[first_month]
    .groupby("age")["loan_personal_cnt"]
    .apply(lambda s: (s > 0).mean())
    .rename("share with a personal loan"),
    title="Personal loans by age (inverted U, peaking mid-30s)",
)

In [ ]:
chances = oracle.filter(like="p_").assign(profile=profile)
chances.groupby("profile").mean().reindex(PROFILES).T.round(4)